## Homework 08: Classification

### Due: July 5th @ 11:59pm (with usual 2 hour + 1 minute grace period)


In this final homework before starting our course project, we will introduce the essential machine learning paradigm of **classification**. We will work with the **UCI Adult** dataset. This is a binary classification task.

As we’ve discussed in this week’s lessons, the classification workflow is similar to what we’ve done for regression, with a few key differences:
- We use `StratifiedKFold` instead of plain `KFold` so that every fold keeps the original class proportions.
- We use classification metrics (e.g., accuracy, precision, recall, F1-score for binary classification) instead of regression metrics.
- We could explore misclassified instances through a confusion matrix (though we will not do that in this homework).

For this assignment, you’ll build a gradient boosting classification using `HistGradientBoostingClassifier` (HGBC) and explore ways of tuning the hyperparameters, including using the technique of early stopping, which basically avoiding have to tune the number of estimators (called `max_iter` in HGBC). 

HGBC has many advantages, which we explain below. 


### Grading

There are 11 graded problems, each worth 5 points. 

In [1]:
# General utilities
import os
import io
import time
import zipfile
import requests
from collections import Counter

# Data handling and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.display import display
 
# Data source
from sklearn.datasets import fetch_openml

 
# scikit-learn core tools 
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    RandomizedSearchCV
)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder

 
# Import model 
from sklearn.ensemble import HistGradientBoostingClassifier
 
# Metrics
from sklearn.metrics import balanced_accuracy_score, classification_report
 
# Distributions for random search
from scipy.stats import loguniform, randint, uniform

# pandas dtypes helpers
from pandas.api.types import is_numeric_dtype, is_categorical_dtype
from pandas import CategoricalDtype

# Optuna Hyperparameter Search tool    (may need to be installed)
import optuna


# Misc

random_seed = 42

def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))



/workspaces/Module-3-Assignments/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Prelude 0: Installing Optuna (if needed)

Optuna is not included in every Python installation. If the statement

```python
import optuna
```

produces

```
ModuleNotFoundError: No module named 'optuna'
```

then install it using one of the following methods.

**If you are using Anaconda (recommended):**

```bash
conda install -c conda-forge optuna
```

**Or using pip:**

```bash
pip install optuna
```

If you are working in a Jupyter notebook, you can also install it directly from a code cell:

```python
%pip install optuna
```

or

```python
!pip install optuna
```

After the installation finishes, **restart the Jupyter kernel** and rerun the notebook from the beginning.

### Prelude 1: Load and Preprocess the UCI Adult Income Dataset

- Load the dataset from sklearn
- Preliminary EDA
- Feature Engineering 

In [2]:
# Load and clean
df = fetch_openml(name='adult', version=2, as_frame=True).frame

df.replace("?", np.nan, inplace=True)            # Some datasets use ? instead of Nan for missing data

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   age             48842 non-null  int64   
 1   workclass       46043 non-null  category
 2   fnlwgt          48842 non-null  int64   
 3   education       48842 non-null  category
 4   education-num   48842 non-null  int64   
 5   marital-status  48842 non-null  category
 6   occupation      46033 non-null  category
 7   relationship    48842 non-null  category
 8   race            48842 non-null  category
 9   sex             48842 non-null  category
 10  capital-gain    48842 non-null  int64   
 11  capital-loss    48842 non-null  int64   
 12  hours-per-week  48842 non-null  int64   
 13  native-country  47985 non-null  category
 14  class           48842 non-null  category
dtypes: category(9), int64(6)
memory usage: 2.7 MB


#### Check: Is the dataset imbalanced?

In [3]:
print(df['class'].value_counts(normalize=True))

class
<=50K    0.760718
>50K     0.239282
Name: proportion, dtype: float64


**YES:** It looks like this dataset is somewhat imbalanced. Therefore, we will 
1. Tell the model to compensate during training by setting `class_weight='balanced'` when defining the model;
2. Evaluate it `balanced_accuracy` instead of `accuracy` and with class-aware metrics (precision, recall, F1); and
3. [Optional] Adjust the probability threshold instead of relying on raw accuracy alone after examining the precision-recall trade-off you observe at 0.5.
    

### Feature Engineering

Based on the considerations in **Appendix One**, we'll make the following changes to the dataset to facilitate training:


1. Drop `fnlwgt` and `education`.   
3. Replace `capital-gain` and `capital-loss` by their difference `capital_net` and add a log-scaled version `capital_net_log`.


In [5]:
# Drop the survey-weight column
df_eng = df.drop(columns=["fnlwgt"])

# Keep only the ordinal education feature
df_eng = df_eng.drop(columns=["education"])      # retain 'education-num'

# Combine capital gains and losses, add a log-scaled variant
df_eng["capital_net"]     = df_eng["capital-gain"] - df_eng["capital-loss"]
df_eng["capital_net_log"] = np.log1p(df_eng["capital_net"].clip(lower=0))
df_eng = df_eng.drop(columns=["capital-gain", "capital-loss"])

# check
df_eng.info()

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   age              48842 non-null  int64   
 1   workclass        46043 non-null  category
 2   education-num    48842 non-null  int64   
 3   marital-status   48842 non-null  category
 4   occupation       46033 non-null  category
 5   relationship     48842 non-null  category
 6   race             48842 non-null  category
 7   sex              48842 non-null  category
 8   hours-per-week   48842 non-null  int64   
 9   native-country   47985 non-null  category
 10  class            48842 non-null  category
 11  capital_net      48842 non-null  int64   
 12  capital_net_log  48842 non-null  float64 
dtypes: category(8), float64(1), int64(4)
memory usage: 2.2 MB


#### Separate target and split

Create the feature set `X` and the target set `y` (using `class` as the target) and split the dataset into 80% training and 20% testing sets, making sure to stratify.

In [6]:

X = df_eng.drop(columns=["class"])
y = (df_eng["class"] == ">50K").astype(int)

# Split (with stratification)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=random_seed,
    stratify=y                           # So same proportion of classes in train and test sets
)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape,  y_test.shape)

Train: (39073, 12) (39073,)
Test : (9769, 12) (9769,)


### Prelude 2: Create a data pipeline and the `HistGradientBoostingClassifier` model

Histogram-based gradient boosting improves on the standard version by:

* **Histogram splits:** bins each feature into ≤ `max_bins` quantiles (i.e., each bin is approximately the same size) and tests splits only between bins, slashing compute time and scaling to large data sets. Default for `max_bins` = 255. 
* **Native NaN handling:** treats missing values as their own bin—no imputation needed.
* **Native Categorical Support**: accepts integer-encoded categories directly and tests “category c vs. all others” splits, eliminating one-hot blow-ups and fake orderings.
* **Built-in early stopping:** stops training after no improvement in validation loss after `n_iter_no_change` rounds. `tol` defines "improvement" (default is 1e-7). 
* **Leaf shrinkage:** adds `l2_regularization`, which ridge-shrinks each leaf value (without changing tree shape) so tiny, noisy leaves have less effect.

>**Summary:**  Histogram-based GB trades a tiny approximation error (binning) for a **huge speed-up** and adds extra conveniences, making it the preferred choice for large tabular data sets. Tuning workflow relies on **Early stopping** to stop training before overfitting occurs. 

In [8]:
# Define a baseline model 

HGBC_model = HistGradientBoostingClassifier(
    # tree structure and learning rate
    learning_rate=0.1,            # These 5 parameters are at defaults for our baseline training in Problem 1             
    max_leaf_nodes=31,            # but will be tuned by randomized search in Problem 2 and Optuna in Problem 3               
    max_depth=None,               
    min_samples_leaf=20,          
    l2_regularization=0.0,        

    # bins and iteration
    max_bins=255,                 # default
    max_iter=500,                 # high enough for early stopping
    early_stopping=True,
    n_iter_no_change=20,
    validation_fraction=0.2,      # 20% monitored for early stopping
    tol=1e-7,                     # default tolerance for validation improvement

    # class imbalance
    class_weight="balanced",

    random_state=random_seed,
    verbose=0
)


### Create a pipeline appropriate for HGBC 

**Why use a `Pipeline` instead of encoding in the dataset first?**

* **Avoid data leakage.** In each CV fold, the `OrdinalEncoder` is refit only on that fold’s training data, so the validation split never influences the encoder.
* **Single, reusable object.** The pipeline bundles preprocessing + model, letting you call `fit`/`predict` on raw data anywhere (CV, Optuna, production) with identical behavior.
* **Compatible with search tools.** `cross_validate`, `GridSearchCV`, and Optuna expect an estimator that can be cloned and refit; a pipeline meets that requirement automatically.

Put simply, the pipeline gives you leak-free evaluation and portable, hassle-free tuning without extra code.


In [9]:
enc = OrdinalEncoder(
    handle_unknown="use_encoded_value",   # Allow unseen categories during transform
    unknown_value=-1,                     # Code for unseen categories
    encoded_missing_value=-2,             # Code for missing values (NaN)
    dtype=np.int64                        # Needed for HistGradientBoostingClassifier
)

# Categorical features
cat_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

# Numeric features (everything that isn’t object / category)
num_cols = X.select_dtypes(include=["number"]).columns.tolist()

preprocess = ColumnTransformer(
    [("cat", enc, cat_cols),
     ("num", "passthrough", num_cols)]
)

pipelined_model = Pipeline([
    ("prep", preprocess),
    ("gb",   HGBC_model)
])

## Problem 1: Baseline Cross-Validation with F1

In this problem, you will run a baseline cross-validation evaluation of your `HistGradientBoostingClassifier` pipeline, using `HGBC_model` defined above. 

**Background:**

* Since the Adult dataset is imbalanced (about 24% positives, 76% negatives), accuracy alone is not reliable.
* We will use the **F1 score** as the evaluation metric, since it balances precision (avoiding false positives) and recall (avoiding false negatives) in a single measure. This is a fairer metric for imbalanced classification, where both types of error matter.
* We will apply **5-fold stratified cross-validation** to make sure each fold has the same proportion of the classes as the original dataset.
* Repeated cross-validation is optional and not required here, because the Adult dataset is large and `HistGradientBoostingClassifier` is robust to small sampling differences. 

**Instructions:**

1. Set up a `StratifiedKFold` cross-validation object with 5 splits, shuffling enabled, and `random_state=random_seed`.
2. Use `cross_val_score` to estimate the mean F1 score and its standard deviation across the folds.
3. Print out the mean and standard deviation of the F1 score, rounded to 4 decimal places.
4. Answer the graded question.


In [10]:
# Your code here
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_seed)

In [11]:
f1_scores = cross_val_score(pipelined_model,
                            X_train, y_train,
                            scoring= 'f1',
                            cv=cv)

In [12]:
f1_scores

array([0.71761542, 0.71247818, 0.71314031, 0.7119448 , 0.7065073 ])

In [13]:
mean_f1 = f1_scores.mean()
std_f1 = f1_scores.std()

print(f"Mean F1: {mean_f1:.4f}")
print(f"Std F1:  {std_f1:.4f}")

Mean F1: 0.7123
Std F1:  0.0035


### Problem 1 Graded Answer

Set `a1` to the mean F1 score of the baseline model. 

In [14]:
 # Your answer here

a1 = mean_f1                    # replace 0 with an expression

In [15]:
# DO NOT change this cell in any way

print(f'a1 = {a1:.4f}')

a1 = 0.7123


## Problem 2: Hyperparameter Optimization with Randomized Search for F1

In this problem, you will tune your `pipelined_model` using `RandomizedSearchCV` to identify the best combination of tree structure and learning rate parameters that maximize the **F1 score**.

**Background:**
The F1 score is our main metric because it balances precision and recall on an imbalanced dataset. Optimizing hyperparameters for F1 ensures we manage both false positives and false negatives in a single measure.

**Instructions:**

1. Set up a randomized search over the following hyperparameter ranges, using appropriate random-number distributions:

   * `learning_rate` (log-uniform between 1e-3 and 0.3)
   * `max_leaf_nodes` (integer from 16 to 256)
   * `max_depth` (integer from 2 to 10)
   * `min_samples_leaf` (integer from 10 to 200)
   * `l2_regularization` (uniform between 0.0 and 2.0)
2. Use **5-fold stratified cross-validation**, with the same settings as in Problem 1.
3. Start `n_iter` at 10 or 20 to prototype, but try for 50 - 100 trials. More trials will generally yield better results, if your time and machine allow.
4. After running the search, show a neatly formatted table of the top 5 results, using `display(...)` showing their mean F1 scores, standard deviation, and the chosen hyperparameter values.
5. Answer the graded question.




In [16]:
# Your code here
pipelined_model.named_steps.keys()


dict_keys(['prep', 'gb'])

In [17]:
param_distributions = {
    'gb__learning_rate': loguniform(1e-3, 3e-1),
    'gb__max_leaf_nodes': randint(16, 257),
    'gb__max_depth': randint(2, 11),
    'gb__min_samples_leaf': randint(10, 201),
    'gb__l2_regularization': [0.0, 2.0],
}

In [18]:
search = RandomizedSearchCV(
    estimator=pipelined_model,
    param_distributions=param_distributions,
    n_iter=50,
    scoring='f1',
    n_jobs=-1,
    cv=cv,
    random_state=random_seed,
)

In [19]:
search.fit(X_train, y_train)

cv_results = pd.DataFrame(search.cv_results_)

In [20]:
score_cols = ['mean_test_score', 'std_test_score', 'rank_test_score']
param_cols = [c for c in cv_results.columns if c.startswith('param_')]
top = (
    cv_results[score_cols + param_cols]
    .sort_values('rank_test_score')
    .head(5)
)

#Rounding of scores
top['mean_test_score'] = top['mean_test_score'].round(4)
top['std_test_score'] = top['std_test_score'].round(4)

rename_map = {c: c.replace('param_', '') for c in param_cols}
top.rename(columns=rename_map, inplace=True)

display(top)

,mean_test_score,std_test_score,rank_test_score,gb__l2_regularization,gb__learning_rate,gb__max_depth,gb__max_leaf_nodes,gb__min_samples_leaf
21,0.7118,0.0028,1,0.0,0.127541,3,48,57
19,0.7111,0.0039,2,0.0,0.056360,4,178,42
11,0.7109,0.0026,3,2.0,0.074323,7,69,115
8,0.7109,0.0045,4,0.0,0.246597,3,24,99
3,0.7107,0.0020,5,0.0,0.252688,7,145,197


In [21]:
max(top['mean_test_score'])

0.7118

### Problem 2 Graded Answer

Set `a2` to the mean F1 score of the best model found. 

In [22]:
 # Your answer here

a2 = max(top['mean_test_score'])                     # replace 0 with your answer, may copy from the displayed results

In [23]:
# DO NOT change this cell in any way

print(f'a2 = {a2:.4f}')

a2 = 0.7118


## Problem 3: Hyperparameter Optimization with Optuna for F1

In this problem, you will explore **Optuna**, a powerful hyperparameter optimization framework, to identify the best combination of hyperparameters that maximize the F1 score of your `pipelined_model`.

**Background:**
Optuna uses a smarter sampling strategy than grid search or randomized search, allowing you to explore the hyperparameter space more efficiently. It also supports *pruning*, which can stop unpromising trials early to save time. This makes it a popular SOTA optimization tool.

**Before you start** browse the [Optuna documentation](https://optuna.org) and view the [tutorial video](https://optuna.readthedocs.io/en/stable/tutorial/index.html). 

As before, we focus on the **F1 score** because it balances precision and recall, making it more robust on an imbalanced dataset.

**Instructions:**

1. Define an Optuna objective function to optimize F1 score, sampling the exact same hyperparameter ranges you did in Problem 2 and using the same CV settings.  
3. Set up an Optuna study with a reasonable number of trials (e.g., up to 100 depending on runtime resources--on my machine Optuna runs about 10x faster than randomized search for the same number of trials, but YMMV).
4. After running the optimization, `display` a clean table with the top 5 trials showing their F1 scores and corresponding hyperparameter settings.
5. Answer the graded question. 

**Note:**  There are many resources on Optuna you can find on the web, but for this problem, you have my permission to let ChatGPT write the code for you. 

In [24]:
# Your code here

pipelined_model.named_steps

{'prep': ColumnTransformer(transformers=[('cat',
                                  OrdinalEncoder(dtype=<class 'numpy.int64'>,
                                                 encoded_missing_value=-2,
                                                 handle_unknown='use_encoded_value',
                                                 unknown_value=-1),
                                  ['workclass', 'marital-status', 'occupation',
                                   'relationship', 'race', 'sex',
                                   'native-country']),
                                 ('num', 'passthrough',
                                  ['age', 'education-num', 'hours-per-week',
                                   'capital_net', 'capital_net_log'])]),
 'gb': HistGradientBoostingClassifier(class_weight='balanced', early_stopping=True,
                                max_iter=500, n_iter_no_change=20,
                                random_state=42, validation_fraction=0.2)}

In [25]:
from sklearn.base import clone

In [26]:
def objective(trial):

    STEP = "gb"

    lr     = trial.suggest_float("learning_rate", 1e-3, 0.3, log=True)
    nleaf  = trial.suggest_int("max_leaf_nodes", 16, 256)
    depth  = trial.suggest_int("max_depth", 2, 10)
    minlf  = trial.suggest_int("min_samples_leaf", 10, 200)
    l2reg  = trial.suggest_float("l2_regularization", 0.0, 2.0)

    params = {
        f"{STEP}__learning_rate":     lr,
        f"{STEP}__max_leaf_nodes":    nleaf,
        f"{STEP}__max_depth":         depth,
        f"{STEP}__min_samples_leaf":  minlf,
        f"{STEP}__l2_regularization": l2reg,
    }

    model = clone(pipelined_model).set_params(**params)

    scores = cross_val_score(
        model, 
        X_train, 
        y_train,
        scoring='f1',
        cv=cv,
        n_jobs=-1
    )

    return scores.mean()

In [27]:
# Optuna Study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, show_progress_bar=True)

[I 2026-07-06 00:45:49,549] A new study created in memory with name: no-name-e03dc7a0-741d-4f76-bf42-ec913507627f
Best trial: 0. Best value: 0.691364:   2%|▏         | 1/50 [00:18<15:26, 18.90s/it]

[I 2026-07-06 00:46:08,451] Trial 0 finished with value: 0.6913640713922881 and parameters: {'learning_rate': 0.007074289217324138, 'max_leaf_nodes': 229, 'max_depth': 8, 'min_samples_leaf': 52, 'l2_regularization': 1.7375240056144603}. Best is trial 0 with value: 0.6913640713922881.


Best trial: 0. Best value: 0.691364:   4%|▍         | 2/50 [00:26<09:50, 12.30s/it]

[I 2026-07-06 00:46:16,131] Trial 1 finished with value: 0.6531485519826147 and parameters: {'learning_rate': 0.003148325200393991, 'max_leaf_nodes': 52, 'max_depth': 3, 'min_samples_leaf': 99, 'l2_regularization': 0.6360437198105051}. Best is trial 0 with value: 0.6913640713922881.


Best trial: 2. Best value: 0.692401:   6%|▌         | 3/50 [00:38<09:24, 12.02s/it]

...

[I 2026-07-06 00:51:15,842] Trial 47 finished with value: 0.626284957759533 and parameters: {'learning_rate': 0.001961302949391654, 'max_leaf_nodes': 153, 'max_depth': 3, 'min_samples_leaf': 26, 'l2_regularization': 1.5357008156128544}. Best is trial 24 with value: 0.7126002397736777.


Best trial: 24. Best value: 0.7126:  98%|█████████▊| 49/50 [05:33<00:06,  6.42s/it]

[I 2026-07-06 00:51:23,381] Trial 48 finished with value: 0.712367891164096 and parameters: {'learning_rate': 0.05093759214602354, 'max_leaf_nodes': 171, 'max_depth': 4, 'min_samples_leaf': 13, 'l2_regularization': 1.0446850878553948}. Best is trial 24 with value: 0.7126002397736777.


Best trial: 24. Best value: 0.7126: 100%|██████████| 50/50 [05:41<00:00,  6.84s/it]

[I 2026-07-06 00:51:31,487] Trial 49 finished with value: 0.7108310855579171 and parameters: {'learning_rate': 0.04224490121758625, 'max_leaf_nodes': 214, 'max_depth': 4, 'min_samples_leaf': 15, 'l2_regularization': 0.7589371489728478}. Best is trial 24 with value: 0.7126002397736777.


In [28]:
rows = []
for t in study.trials:
    if t.state == optuna.trial.TrialState.COMPLETE:
        row = {"trial": t.number, "mean_f1": t.value}
        row.update(t.params)
        rows.append(row)

top5 = pd.DataFrame(rows).sort_values("mean_f1", ascending=False).head(5).copy()
top5["mean_f1"] = top5["mean_f1"].round(4)
display(top5)

,trial,mean_f1,learning_rate,max_leaf_nodes,max_depth,min_samples_leaf,l2_regularization
24,24,0.7126,0.184715,139,3,149,1.301925
31,31,0.7125,0.198200,156,3,158,1.803946
48,48,0.7124,0.050938,171,4,13,1.044685
38,38,0.7121,0.141039,208,2,10,1.881007
29,29,0.7121,0.291790,159,2,172,1.761664


In [29]:
print(f"Best mean F1 (CV): {study.best_value:.4f}")
print("Best hyperparameters:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

Best mean F1 (CV): 0.7126
Best hyperparameters:
  learning_rate: 0.18471479744241412
  max_leaf_nodes: 139
  max_depth: 3
  min_samples_leaf: 149
  l2_regularization: 1.3019254445577968


### Problem 3 Graded Answer

Set `a3` to the mean F1 score of the best model found. 

In [30]:
 # Your answer here

a3 = study.best_value                    # replace 0 with your answer, may copy from the displayed results

In [31]:
# DO NOT change this cell in any way

print(f'a3 = {a3:.4f}')

a3 = 0.7126


## Problem 4: Final Model Evaluation on Test Set

In this problem, you will take the best hyperparameter configuration you found in your earlier experiments (Randomized Search or Optuna) and fully evaluate the resulting model on the test set.

**Background:**
When performing hyperparameter tuning, we typically optimize for a single metric (e.g., F1). However, before deployment, it is essential to check **all relevant metrics** on the final test set to understand the model’s behavior in a balanced way.

**Instructions:**

1. Take the best hyperparameters you found in Problems 2 or 3 and apply them to your `pipelined_model`.
2. Re-train this final tuned model on the **entire training set** (not just the folds).
3. Evaluate the final model on the heldout **test set**, reporting the following metrics:

   * Precision
   * Recall
   * F1 score
   * Balanced accuracy
4. Use `classification_report` **on the test set** to print precision, recall, and F1 score, and use `balanced_accuracy_score` separately to calculate and print balanced accuracy.
5. Answer the graded questions.

**Note:** We evaluate the metrics on the test set because it was never seen during training or hyperparameter tuning. This gives us an unbiased estimate of how the model will perform on truly unseen data. Evaluating on the training set would be misleading, because the model has already learned from that data and could appear artificially good.


In [32]:
# Your code here
best_params = {
    "gb__learning_rate": 0.18471479744241412,
    "gb__max_leaf_nodes": 139,
    "gb__max_depth": 3,
    "gb__min_samples_leaf": 149,
    "gb__l2_regularization": 1.3019254445577968
}

best_model = clone(pipelined_model).set_params(**best_params)

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

print("=== Classification Report (Test Set) ===")
report = classification_report(y_test, y_pred, digits=4, output_dict=True)
df_report = pd.DataFrame(report).T
print(df_report)

bal_acc = balanced_accuracy_score(y_test, y_pred)
print(f"Balanced Accuracy (Test Set): {bal_acc:.4f}")


=== Classification Report (Test Set) ===
              precision    recall  f1-score    support
0              0.952034  0.825326  0.884163  7431.0000
1              0.609859  0.867836  0.716328  2338.0000
accuracy       0.835500  0.835500  0.835500     0.8355
macro avg      0.780946  0.846581  0.800246  9769.0000
weighted avg   0.870141  0.835500  0.843996  9769.0000
Balanced Accuracy (Test Set): 0.8466


In [33]:
report['macro avg']['precision']

0.7809461307748304

### Problem 4 Graded Questions

- Set `a4a` to the balanced accuracy score of the best model.
- Set `a4b` to the macro average precision of this model.
- Set `a4c` to the macro average recall score of the this model.

**Note:** Macro average takes the mean of each class’s precision/recall without considering how many samples each class has, which is appropriate for a balanced evaluation.

In [34]:
 # Your answer here

a4a = bal_acc                     # replace 0 with your answer, use variable or expression from above

In [35]:
# DO NOT change this cell in any way

print(f'a4a = {a4a:.4f}')

a4a = 0.8466


In [36]:
 # Your answer here

a4b = report['macro avg']['precision']                    # replace 0 with your answer, may copy from the displayed results

In [37]:
# DO NOT change this cell in any way

print(f'a4b = {a4b:.4f}')

a4b = 0.7809


In [38]:
 # Your answer here

a4c = report['macro avg']['recall']                    # replace 0 with your answer, may copy from the displayed results

In [39]:
# DO NOT change this cell in any way

print(f'a4c = {a4c:.4f}')

a4c = 0.8466


## **Problem Five: Analytical Questions**

Select the single **best** answer to each of the following multiple choice questions on this week's material.

You may wish to review Appendix One for a brief tutorial on the various classification metrics. 

### Part A: Tests - Precision, Recall, and F1 in Practice

A bank uses your model to identify customers earning over $50K for a premium product invitation. Based on your final test set evaluation, including macro-averaged precision and recall, which of the following best describes what might happen?

1. The bank will miss some eligible high-income customers, but will avoid marketing mistakes by sending invitations only to those it is  confident about.

2. The bank will successfully reach most high-income customers, but will also waste resources sending invitations to some low-income customers.

3. The bank will perfectly identify all high-income and low-income customers, resulting in no wasted invitations and no missed opportunities.

4. The bank will correctly identify most high-income customers while also sending invitations to very few low-income customers, because a high F1 score guarantees both high precision and high recall.

In [40]:
# Your answer here. 

a5a = 4                    # Must be one of 1, 2, 3, 4

In [41]:
### Do not change this cell in any way

print(f'a5a = {a5a}')

a5a = 4


### Part B: Why Use Stratified Cross-Validation?

Why is **StratifiedKFold** preferred over ordinary **KFold** for this homework?

1. It guarantees every fold has exactly the same number of observations.
2. It keeps approximately the same class proportions in every fold.
3. It prevents overfitting by using more training samples.
4. It automatically balances the training data by duplicating minority-class examples.

In [42]:
# Your answer here. 

a5b = 2                    # Must be one of 1, 2, 3, 4

In [43]:
# Do not change this cell in any way

print(f'a5b = {a5b}')

a5b = 2


### Part C: Understanding Precision and Recall

Suppose you lower the probability threshold used to classify someone as earning more than $50K.

Which change is **most likely**?

1. Precision increases while recall decreases.
2. Both precision and recall increase.
3. Recall increases while precision decreases.
4. Precision, recall, and F1 always remain unchanged.

In [44]:
# Your answer here. 

a5c = 3                    # Must be one of 1, 2, 3, 4

In [45]:
# Do not change this cell in any way

print(f'a5c = {a5c}')

a5c = 3


### Part D: Why Use a Pipeline?

The homework places the `OrdinalEncoder` inside a `Pipeline` instead of encoding the entire dataset before cross-validation.

What is the primary reason?

1. Pipelines train the classifier faster.
2. Pipelines reduce the number of hyperparameters.
3. Pipelines prevent data leakage during cross-validation.
4. Pipelines automatically improve F1 score.

In [46]:
# Your answer here. 

a5d = 3                    # Must be one of 1, 2, 3, 4

In [47]:
# Do not change this cell in any way

print(f'a5d = {a5d}')

a5d = 3


### Part E: Early Stopping

`HistGradientBoostingClassifier` is configured with a large value of `max_iter` while enabling **early stopping**.

Why is this a good strategy?

1. It guarantees the model reaches the maximum number of iterations.
2. It allows training to stop automatically once additional iterations no longer improve validation performance.
3. It eliminates the need for cross-validation.
4. It ensures the model never overfits.

In [48]:
# Your answer here. 

a5e = 2                   # Must be one of 1, 2, 3, 4

In [49]:
# Do not change this cell in any way

print(f'a5e = {a5e}')

a5e = 2


### Appendix One: Understanding Precision, Recall, F1, and Balanced Accuracy

**Tutorial**

In binary classification, you will often evaluate these key metrics:

* **Precision**: *Of all the positive predictions the model made, how many were actually correct?*

  * High precision = few false positives
  * Low precision = many false positives

* **Recall**: *Of all the actual positive cases, how many did the model correctly identify?*

  * High recall = few false negatives
  * Low recall = many false negatives

* **F1 score**: The harmonic mean of precision and recall, which balances them in a single measure.

  * F1 is **highest** when precision and recall are both high and similar in value.
  * If precision and recall are unbalanced, F1 will drop to reflect that imbalance.

* **Balanced accuracy**: The average of recall across both classes (positive and negative).

  * It ensures the classifier is performing reasonably well on *both* groups, correcting for class imbalance.
  * Balanced accuracy is especially important if the classes are very unequal in size.

**Typical trade-offs to remember:**

* **Higher recall, lower precision**: the model finds most true positives but also mislabels some negatives as positives
* **Higher precision, lower recall**: the model is strict about positive predictions, but misses some true positives
* **Balanced precision and recall (good F1)**: a practical compromise
* **Balanced accuracy**: checks fairness across both classes

### Appendix Two: Feature Engineering

Here are some practical feature-engineering tweaks worth considering (beyond simply ordinal-encoding the categoricals)

| Feature(s)                                                           | Why the tweak can help                                                                                                                                                     | How to do it (quick version)                                                                                                                                                    | Keep / drop?      |
| -------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------- |
| **`fnlwgt`**                                                         | Survey sampling weight, not a predictor. Leaving it in often lets the model “cheat.”                                                                                       | `df = df.drop(columns=["fnlwgt"])`                                                                                                                                              | **Drop**          |
| **`education` *vs.* `education-num`**                                | They encode the **same** information twice (categorical label and its ordinal rank). Keeping both is redundant and can cause leakage of a perfectly predictive feature.    | Usually keep **only one**. For tree models `education-num` is simplest: `df = df.drop(columns=["education"])`                                                                   | **Drop one**      |
| **`capital-gain`, `capital-loss`**                                   | Highly skewed; most values are zero with a long upper tail. The sign (gain vs. loss) matters, but treating them separately wastes a feature slot.                          | 1) Combine: `df["capital_net"] = df["capital-gain"] - df["capital-loss"]`; 2) Log-transform to reduce skew: `df["capital_net_log"] = np.log1p(df["capital_net"].clip(lower=0))` | Replace originals |
| **`age`, `hours-per-week`**                                          | Continuous but with natural plateaus—trees handle splits fine, yet log or square-root scaling can soften extreme values; bucketing makes partial-dependence plots clearer. | Simple bucket: `df["age_bin"] = pd.cut(df["age"], bins=[16,25,35,45,55,65,90])` (optional)                                                                                      | Optional          |
| **Missing categories** (`workclass`, `occupation`, `native-country`) | HGB handles `-1`/`-2` codes fine, but you may want *explicit* “Missing” bucket for interpretability.                                                                       | Use `encoded_missing_value=-2` during encoding.                                                                                                            | Keep as is        |
| **Rare categories in `native-country`**                              | Hundreds of low-frequency countries dilute signal; grouping boosts stability.                                                                                              | Map infrequent categories to “Other”:                                                                                                                                           |                   |


#### Minimum set of tweaks (good baseline, low effort)

1. **Drop `fnlwgt`.**  
2. **Keep `education-num`, drop `education`.**  
3. **Combine `capital-gain` and `capital-loss` into `capital_net`** (optionally add a log-scaled version).  
4. Leave other numeric/categorical features as is; your histogram-GBDT will cope.


